In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalogo", "catalog_dev")
catalogo = dbutils.widgets.get("catalogo").strip()
assert catalogo in ("catalog_dev", "catalog_prod"), "Catálogo no permitido"

def texto(nombre):
    valor = F.trim(F.col(nombre))
    return F.when(valor != "", valor).otherwise(F.lit(None).cast("string"))

def codigo_postal(nombre):
    valor = texto(nombre)
    return F.when(valor.isNotNull(), F.lpad(valor, 5, "0")).otherwise(F.lit(None).cast("string"))

def fecha_hora(nombre):
    return F.to_timestamp(F.col(nombre), "yyyy-MM-dd HH:mm:ss")

def guardar_silver(df, nombre, esperado=None):
    destino = f"{catalogo}.silver.{nombre}"
    columnas_destino = spark.table(destino).columns
    faltantes = set(columnas_destino) - set(df.columns)
    adicionales = set(df.columns) - set(columnas_destino)
    if faltantes or adicionales:
        raise ValueError(f"{nombre}: faltantes={faltantes}, adicionales={adicionales}")
    # insertInto respeta el orden de columnas, no sus nombres; alineamos expresamente.
    df.select(*columnas_destino).write.mode("overwrite").insertInto(destino)
    total = spark.table(destino).count()
    if esperado is not None and total != esperado:
        raise ValueError(f"{nombre}: esperado={esperado}, obtenido={total}")
    print(f"OK | {destino}: {total:,} registros")

print("Silver configurado para", catalogo)

In [0]:
## PEDIDOS:
origen = spark.table(f"{catalogo}.bronze.orders")

df_pedidos = origen.select(
    texto("order_id").alias("id_pedido"),
    texto("customer_id").alias("id_cliente"),
    F.lower(texto("order_status")).alias("estado_pedido"),
    fecha_hora("order_purchase_timestamp").alias("fecha_compra"),
    fecha_hora("order_approved_at").alias("fecha_aprobacion"),
    fecha_hora("order_delivered_carrier_date").alias("fecha_entrega_transportista"),
    fecha_hora("order_delivered_customer_date").alias("fecha_entrega_cliente"),
    fecha_hora("order_estimated_delivery_date").alias("fecha_entrega_estimada")
)

guardar_silver(df_pedidos, "pedidos", 99441)

In [0]:
## ARTICULOS:
origen = spark.table(f"{catalogo}.bronze.order_items")

df_detalle = origen.select(
    texto("order_id").alias("id_pedido"),
    F.col("order_item_id").cast("int").alias("id_item"),
    texto("product_id").alias("id_producto"),
    texto("seller_id").alias("id_vendedor"),
    fecha_hora("shipping_limit_date").alias("fecha_limite_envio"),
    F.col("price").cast("decimal(18,2)").alias("precio"),
    F.col("freight_value").cast("decimal(18,2)").alias("valor_flete")
)

guardar_silver(df_detalle, "detalle_ventas", 112650)

In [0]:
## CLIENTES:
origen = spark.table(f"{catalogo}.bronze.customers")

df_clientes = origen.select(
    texto("customer_id").alias("id_cliente"),
    texto("customer_unique_id").alias("id_cliente_unico"),
    codigo_postal("customer_zip_code_prefix").alias("codigo_postal"),
    F.lower(texto("customer_city")).alias("ciudad"),
    F.upper(texto("customer_state")).alias("estado")
)

guardar_silver(df_clientes, "clientes", 99441)

In [0]:
## PRODUCTOS:
origen = spark.table(f"{catalogo}.bronze.products")

df_productos = origen.select(
    texto("product_id").alias("id_producto"),
    F.coalesce(texto("product_category_name"), F.lit("SIN CLASIFICAR")).alias("categoria"),
    F.col("product_name_lenght").cast("int").alias("longitud_nombre"),
    F.col("product_description_lenght").cast("int").alias("longitud_descripcion"),
    F.col("product_photos_qty").cast("int").alias("cantidad_fotos"),
    F.col("product_weight_g").cast("double").alias("peso_g"),
    F.col("product_length_cm").cast("double").alias("largo_cm"),
    F.col("product_height_cm").cast("double").alias("alto_cm"),
    F.col("product_width_cm").cast("double").alias("ancho_cm")
)

guardar_silver(df_productos, "productos", 32951)

In [0]:
# VENDEDORES:
origen = spark.table(f"{catalogo}.bronze.sellers")

df_vendedores = origen.select(
    texto("seller_id").alias("id_vendedor"),
    codigo_postal("seller_zip_code_prefix").alias("codigo_postal"),
    F.lower(texto("seller_city")).alias("ciudad"),
    F.upper(texto("seller_state")).alias("estado")
)

guardar_silver(df_vendedores, "vendedores", 3095)

In [0]:
## PAGOS:
origen = spark.table(f"{catalogo}.bronze.order_payments")

df_pagos = origen.select(
    texto("order_id").alias("id_pedido"),
    F.col("payment_sequential").cast("int").alias("secuencia_pago"),
    F.lower(texto("payment_type")).alias("metodo_pago"),
    F.col("payment_installments").cast("int").alias("cuotas"),
    F.col("payment_value").cast("decimal(18,2)").alias("importe_pago")
)

guardar_silver(df_pagos, "pagos", 103886)

In [0]:
## RESEÑAS:
origen = spark.table(f"{catalogo}.bronze.order_reviews")

df_resenas = origen.select(
    texto("review_id").alias("id_resena"),
    texto("order_id").alias("id_pedido"),
    F.col("review_score").cast("int").alias("calificacion"),
    texto("review_comment_title").alias("titulo"),
    texto("review_comment_message").alias("comentario"),
    fecha_hora("review_creation_date").alias("fecha_creacion"),
    fecha_hora("review_answer_timestamp").alias("fecha_respuesta")
)

guardar_silver(df_resenas, "resenas", 99224)

In [0]:
## CATEGORIAS:
origen = spark.table(f"{catalogo}.bronze.category_translation")

df_categorias = origen.select(
    texto("product_category_name").alias("categoria"),
    texto("product_category_name_english").alias("categoria_ingles")
)

guardar_silver(df_categorias, "categorias", 71)

In [0]:
geo_origen = spark.table(f"{catalogo}.bronze.geolocation")

geo = geo_origen.select(
    codigo_postal("geolocation_zip_code_prefix").alias("codigo_postal"),
    F.lower(texto("geolocation_city")).alias("ciudad"),
    F.upper(texto("geolocation_state")).alias("estado"),
    F.col("geolocation_lat").cast("double").alias("latitud"),
    F.col("geolocation_lng").cast("double").alias("longitud")
).filter(F.col("codigo_postal").isNotNull())

coordenadas_validas = F.col("latitud").between(-34, 6) & F.col("longitud").between(-74, -28)
geo_coordenadas = (
    geo.withColumn("latitud", F.when(coordenadas_validas, F.col("latitud")))
       .withColumn("longitud", F.when(coordenadas_validas, F.col("longitud")))
       .groupBy("codigo_postal")
       .agg(
           F.percentile_approx("latitud", 0.5, 10000).alias("latitud"),
           F.percentile_approx("longitud", 0.5, 10000).alias("longitud"),
           F.count("*").cast("bigint").alias("registros_geograficos")
       )
)

ref_clientes = spark.table(f"{catalogo}.silver.clientes").select(
    "codigo_postal", "ciudad", "estado"
).withColumn("prioridad", F.lit(1))
ref_vendedores = spark.table(f"{catalogo}.silver.vendedores").select(
    "codigo_postal", "ciudad", "estado"
).withColumn("prioridad", F.lit(2))
ref_geo = geo.select("codigo_postal", "ciudad", "estado").withColumn("prioridad", F.lit(3))

candidatos = (
    ref_clientes.unionByName(ref_vendedores).unionByName(ref_geo)
    .filter(F.col("codigo_postal").isNotNull())
)

frecuencias = candidatos.groupBy("codigo_postal", "ciudad", "estado", "prioridad").count()
orden = Window.partitionBy("codigo_postal").orderBy(
    F.col("prioridad").asc(), F.col("count").desc(),
    F.col("estado").asc_nulls_last(), F.col("ciudad").asc_nulls_last()
)
ubicaciones_representativas = (
    frecuencias.withColumn("fila", F.row_number().over(orden))
    .filter(F.col("fila") == 1)
    .select("codigo_postal", "ciudad", "estado")
)

df_ubicaciones = (
    ubicaciones_representativas
    .join(geo_coordenadas, "codigo_postal", "left")
    .select(
        "codigo_postal", "ciudad", "estado", "latitud", "longitud",
        F.coalesce(F.col("registros_geograficos"), F.lit(0).cast("bigint")).alias("registros_geograficos")
    )
)

guardar_silver(df_ubicaciones, "ubicaciones")